In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

try:
    X_final_cleaned = pd.read_csv('medical_conditions_larger_mildly_imbalanced.csv')
    print("Data 'X_final_cleaned' loaded successfully.")
except FileNotFoundError:
    print("Error: 'medical_conditions_larger_mildly_imbalanced.csv' not found. Please ensure the file is in the correct directory.")
    exit()

X = X_final_cleaned.drop(columns=['condition'])
y = X_final_cleaned['condition']
categorical_features = ['gender', 'smoking_status']

for col in categorical_features:
    if col in X.columns and X[col].dtype != 'object':
        X[col] = X[col].astype('object')
        
#splitting model
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# CatBoost Model with fine tuned hyperparameters
cat_model_tuned = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    loss_function='MultiClass',
    eval_metric='Accuracy', 
    custom_metric=['F1'], 
    random_state=42,
    verbose=0,
    cat_features=categorical_features,
    auto_class_weights='Balanced',
    depth=8, 
    #increased L2 regularization to control overfitting
    l2_leaf_reg=5
)

print("\nStarting CatBoost fine-tuning with increased iterations, depth, and L2 regularization...")

# Increased patience for early stopping
cat_model_tuned.fit(X_train, y_train, early_stopping_rounds=40)
#evaluation
cat_y_pred_tuned = cat_model_tuned.predict(X_test)
# Flatten prediction array if necessary
if isinstance(cat_y_pred_tuned, np.ndarray) and cat_y_pred_tuned.ndim > 1:
    cat_y_pred_tuned = cat_y_pred_tuned.flatten()

# final metrics
cat_accuracy_tuned = accuracy_score(y_test, cat_y_pred_tuned)
cat_f1_score_tuned = f1_score(y_test, cat_y_pred_tuned, average='weighted')
cat_class_report_tuned = classification_report(y_test, cat_y_pred_tuned)

print("CatBoost model fine-tuning completed")
print(f"\n--- CatBoost performance on test set (Fine-Tuned) ---")
print(f"Overall Accuracy: **{cat_accuracy_tuned:.4f}**")
print(f"Weighted F1 Score: **{cat_f1_score_tuned:.4f}**")
print("\nClassification report (Tuned Model):\n", cat_class_report_tuned)

Data 'X_final_cleaned' loaded successfully.

Starting CatBoost fine-tuning with increased iterations, depth, and L2 regularization...
CatBoost model fine-tuning completed

--- CatBoost performance on test set (Fine-Tuned) ---
Overall Accuracy: **0.9181**
Weighted F1 Score: **0.9186**

Classification report (Tuned Model):
               precision    recall  f1-score   support

      Cancer       0.92      0.93      0.93      1200
    Diabetic       0.99      0.87      0.93      1804
   Pneumonia       0.85      0.97      0.90      1500

    accuracy                           0.92      4504
   macro avg       0.92      0.92      0.92      4504
weighted avg       0.93      0.92      0.92      4504

